# E-commerce: RFM-сегментация и анализ лояльности клиентов

**Бизнес-задача:** На основе выгрузки из реляционной базы данных интернет-магазина (более 100 000 строк) собрать единую витрину данных (Data Mart). Провести RFM-анализ (Recency, Frequency, Monetary) для выявления самых ценных клиентов и сегмента "уходящих VIP-пользователей", чтобы передать данные в отдел маркетинга для реактивации.

## 1. Загрузка данных и объединение таблиц (ETL)
В реляционных базах данные хранятся раздельно. Мы загружаем три ключевые таблицы (клиенты, заказы, состав заказов) и объединяем их с помощью `Outer Join`, чтобы не потерять ни одной транзакции.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.float_format', '{:.2f}'.format)

df_customers = pd.read_csv('olist_customers_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')

res = pd.merge(df_customers, df_orders, on='customer_id', how='outer')
df_result = pd.merge(res, df_items, on='order_id', how='outer')
df_result.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,1.00,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,1.00,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,1.00,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,1.00,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,1.00,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [2]:
df_result.shape
df_result.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

## 2. Группировка и поиск аномалий (Frequency & Monetary)
Агрегируем базу по уникальным пользователям (`customer_unique_id`). Находим клиентов, принесших наибольшую выручку (Monetary) и совершивших наибольшее количество заказов (Frequency).

In [3]:
# 1. ГРУППИРОВКА И АГРЕГАЦИЯ (Создаем RFM-базу)
# Считаем сумму ('sum') для цены и количество уникальных ('nunique') для заказов
df_rfm = df_result.groupby('customer_unique_id').agg({
    'price': 'sum',
    'order_id': 'nunique'   
})

# 2. ПОИСК ЧЕМПИОНОВ (Без хардкода!)
# .idxmax() находит индекс (в нашем случае - customer_unique_id) строки с максимальным значением
top_money_id = df_rfm['price'].idxmax()
top_orders_id = df_rfm['order_id'].idxmax()

# 3. ФИЛЬТРАЦИЯ ИСХОДНИКА (Рассматриваем чемпионов под лупой)
# Берем большую склеенную базу и оставляем только строки покупок наших рекордсменов
df_top_sum = df_result[df_result['customer_unique_id'] == top_money_id]
df_top_order = df_result[df_result['customer_unique_id'] == top_orders_id]

# 4. ВЫВОД РЕЗУЛЬТАТОВ (Детализация чека)
print("Детализация чека: Клиент с максимальной суммой:")
print(df_top_sum['price'].value_counts(), "\n")

print("Детализация чека: Клиент с максимальным числом заказов:")
print(df_top_order['price'].value_counts(), "\n")

# 5. КРАСИВЫЙ ВЫВОД СТАТИСТИКИ 
# Используем переменную top_orders_id внутри двойных скобок [[ ]], чтобы получить красивый DataFrame
print("Общая статистика по самому лояльному клиенту:")
print(df_rfm.loc[[top_orders_id]].to_markdown(index=True))

Детализация чека: Клиент с максимальной суммой:
price
1680.00    8
Name: count, dtype: int64 

Детализация чека: Клиент с максимальным числом заказов:
price
13.99     3
39.90     2
99.00     1
45.99     1
69.90     1
14.99     1
51.80     1
22.99     1
72.90     1
149.90    1
26.99     1
29.99     1
23.40     1
Name: count, dtype: int64 

Общая статистика по самому лояльному клиенту:
| customer_unique_id               |   price |   order_id |
|:---------------------------------|--------:|-----------:|
| 8d50f5eadf50201ccdcedfb9e2ac8455 |  729.62 |         17 |


## 3. Работа с временными рядами (Recency)
Переводим текстовые даты в формат `datetime`. Рассчитываем метрику Recency — количество дней, прошедших с момента последней покупки клиента до "сегодняшнего дня" (даты последней транзакции в базе).

In [4]:
df_result['order_purchase_timestamp'] = pd.to_datetime(df_result['order_purchase_timestamp'])
df_result.dtypes
print("Дата самого первого заказа:")
print(df_result['order_purchase_timestamp'].min())
print("Дата самого последнего заказа:")
print(df_result['order_purchase_timestamp'].max())
now = (df_result['order_purchase_timestamp'].max()) + pd.Timedelta(days=1)
df_time_order = df_result.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
df_time_order['day_last_order'] = (now - df_time_order['order_purchase_timestamp']).dt.days

df_full = pd.merge(df_rfm, df_time_order, on='customer_unique_id', how='outer')
df_full = df_full.drop('order_purchase_timestamp', axis=1)
df_full.columns = ['customer_unique_id', 'Monetary', 'Frequency', 'Recency']

Дата самого первого заказа:
2016-09-04 21:15:19
Дата самого последнего заказа:
2018-10-17 17:30:18


## 4. Бизнес-инсайты: Поиск "Уходящих VIP"
Сегментируем аудиторию. Нас интересуют клиенты, которые принесли компании значительную выручку (Monetary > 2000), но перестали покупать и находятся в зоне риска оттока (Recency > 300 дней).

In [5]:
df_filtr_full = df_full[(df_full['Monetary']>2000) & (df_full['Recency']>300)]
print(f"Количество уходящих VIP клиентов - {len(df_filtr_full)}")
print(f"ТОП по количеству заказов  -  \n{df_full.iloc[df_full['Frequency'].idxmax()].to_markdown(index=True)}")

Количество уходящих VIP клиентов - 63
ТОП по количеству заказов  -  
|                    | 52973                            |
|:-------------------|:---------------------------------|
| customer_unique_id | 8d50f5eadf50201ccdcedfb9e2ac8455 |
| Monetary           | 729.62                           |
| Frequency          | 17                               |
| Recency            | 58                               |


## Выводы
1. **Качество данных:** Успешно проведена консолидация данных из 3 различных таблиц реляционной БД в единую витрину (Data Mart) объемом более 113 000 строк.
2. **Паттерны потребления:** Выявлено фундаментальное различие между "китами" (B2B-оптовиками с крупным разовым чеком) и истинно лояльными клиентами (B2C-покупателями с высокой частотой покупок). Клиент с максимальной выручкой совершил всего 1 заказ, в то время как самый лояльный клиент возвращался 17 раз.
3. **Зона риска оттока (Churn):** Выявлено **63 VIP-клиента** (с чеком >2000 у.е.), которые не совершали покупок более 300 дней. Эти данные передаются в отдел retention-маркетинга для таргетированных рассылок и реактивации, что потенциально может вернуть бизнесу десятки тысяч долларов зависшей выручки.